# Notebook 13 — Business ROI Frame

Translates model outputs into actionable retention campaign economics for KKBox.

In [1]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from pathlib import Path

ROOT    = Path('..').resolve()
OUTPUTS = ROOT / 'outputs'

# Campaign parameters
RETENTION_RATE = 0.25   # 25% of targeted churners retained by offer
OFFER_COST     = 8.0    # USD equiv (one month free or equivalent discount)

print(f'OUTPUTS: {OUTPUTS}')

OUTPUTS: C:\Users\h11la\OneDrive\Documents\00 Portfolio\Customer Analytics ML Pipeline\retail-clv-churn-prediction\outputs


## Load Data and Generate Predictions

In [2]:
def load_split(name):
    df = pd.read_parquet(OUTPUTS / f'{name}.parquet')
    df.columns = [c.replace(' ', '_').replace('-', '_') for c in df.columns]
    return df

test = load_split('test')
print(f'Test shape: {test.shape}')

NON_FEATURE_COLS = {
    'msno', 'is_churn', 'registration_init_time',
    'last_expire_date', 'last_transaction', 'first_transaction',
    'last_log_date', 'first_log_date'
}
FEATURE_COLS = [c for c in test.columns if c not in NON_FEATURE_COLS]
X_test = test[FEATURE_COLS].values
y_test = test['is_churn'].values

# Generate churn probabilities from final model (XGBoost)
final_model = joblib.load(OUTPUTS / 'final_model.pkl')
churn_prob  = final_model.predict_proba(X_test)[:, 1]

# Parse optimal threshold from key=value file
thresh_data = {}
with open(OUTPUTS / 'optimal_threshold.txt') as f:
    for line in f:
        if '=' in line:
            k, v = line.strip().split('=', 1)
            thresh_data[k.strip()] = v.strip()
optimal_threshold = float(thresh_data['optimal_threshold'])
print(f'Optimal threshold: {optimal_threshold}')
print(f'Churn prob — mean: {churn_prob.mean():.4f}, max: {churn_prob.max():.4f}')

Test shape: (125272, 37)


Optimal threshold: 0.45
Churn prob — mean: 0.1508, max: 0.9999


In [3]:
# Build test predictions dataframe
test_preds = pd.DataFrame({
    'msno':              test['msno'].values,
    'actual_churn':      y_test,
    'churn_prob':        churn_prob,
    'predicted_churn':   (churn_prob >= optimal_threshold).astype(int)
})

# Load CLV predictions (non-churned customers only)
clv_preds = pd.read_parquet(OUTPUTS / 'clv_predictions.parquet')
print(f'CLV predictions shape: {clv_preds.shape}')

# LEFT JOIN: churned customers get predicted_clv = 0
df = test_preds.merge(clv_preds[['msno', 'predicted_clv']], on='msno', how='left')
df['predicted_clv'] = df['predicted_clv'].fillna(0.0)

print(f'Merged df shape: {df.shape}')
print(df.head())

CLV predictions shape: (113452, 3)
Merged df shape: (125272, 5)
                                           msno  actual_churn  churn_prob  \
0  O2Zx3FkWQu34G+6Enlvgz6p9/alytpZ4N6X9keE4mgg=             1    0.602401   
1  mIGY+iet/Ls25lZNal87wfFZgbYnlbOruOE90Bhj1zA=             1    0.985868   
2  0zGhnY6ymrVY7/jJoDdGZyLRsLnMGQ12R6+CMhAMi6A=             1    0.990125   
3  hf6rhFzrkZO5tZKtVc6VEjunJ4yKfFA/Oiit+Ybx3fo=             1    0.473255   
4  y7dsv7J/zVulFdhmP2/Lwa5ikeo6gH02STSD9quifRU=             1    0.269020   

   predicted_churn  predicted_clv  
0                1            0.0  
1                1            0.0  
2                1            0.0  
3                1            0.0  
4                0            0.0  


## Section 1 — Target Segment

In [4]:
# Flag predicted churn
df['is_predicted_churn'] = df['predicted_churn']

# Flag high-value: predicted_clv >= 75th percentile of non-zero CLV predictions
clv_nonzero = df.loc[df['predicted_clv'] > 0, 'predicted_clv']
clv_75th    = clv_nonzero.quantile(0.75)
df['is_high_value'] = (df['predicted_clv'] >= clv_75th).astype(int)

# Target segment: predicted to churn AND high-value
df['is_target'] = ((df['is_predicted_churn'] == 1) & (df['is_high_value'] == 1)).astype(int)

target = df[df['is_target'] == 1].copy()

print('=== Target Segment ===')
print(f'Total test customers        : {len(df):,}')
print(f'Predicted churners          : {df["is_predicted_churn"].sum():,}  ({df["is_predicted_churn"].mean()*100:.1f}%)')
print(f'CLV 75th percentile         : ${clv_75th:.2f}')
print(f'Target segment size         : {len(target):,}')
print(f'Mean predicted CLV (target) : ${target["predicted_clv"].mean():.2f}')
print(f'Mean churn prob  (target)   : {target["churn_prob"].mean():.4f}')

=== Target Segment ===
Total test customers        : 125,272
Predicted churners          : 16,309  (13.0%)
CLV 75th percentile         : $2401.23
Target segment size         : 2,869
Mean predicted CLV (target) : $2729.07
Mean churn prob  (target)   : 0.6540


## Section 2 — SaaS-Specific ROI Calculation

In [5]:
# EV per customer = P(churn) * predicted_clv * retention_rate - offer_cost
target = target.copy()
target['ev'] = (
    target['churn_prob'] * target['predicted_clv'] * RETENTION_RATE
    - OFFER_COST
)

n_target            = len(target)
total_campaign_cost = n_target * OFFER_COST
net_roi             = target['ev'].sum()
roi_pct             = net_roi / total_campaign_cost * 100 if total_campaign_cost > 0 else 0

print('=' * 55)
print('  RETENTION CAMPAIGN ROI SUMMARY')
print('=' * 55)
print(f'  Target customers          : {n_target:,}')
print(f'  Offer cost per customer   : ${OFFER_COST:.2f}')
print(f'  Assumed retention rate    : {RETENTION_RATE*100:.0f}%')
print(f'  Total campaign cost       : ${total_campaign_cost:,.0f}')
print(f'  Net ROI                   : ${net_roi:,.0f}')
print(f'  ROI %%                     : {roi_pct:.1f}%%')
print('=' * 55)

  RETENTION CAMPAIGN ROI SUMMARY
  Target customers          : 2,869
  Offer cost per customer   : $8.00
  Assumed retention rate    : 25%
  Total campaign cost       : $22,952
  Net ROI                   : $1,257,503
  ROI %%                     : 5478.8%%


## Section 3 — Sensitivity Analysis

In [6]:
retention_rates = np.arange(0.10, 0.41, 0.05)   # 0.10 … 0.40
offer_costs     = np.arange(4, 21, 2)            # 4, 6, 8 … 20

# Gross retained revenue per customer (before offer cost)
gross_per_customer = (target['churn_prob'] * target['predicted_clv']).values

heatmap_data = np.zeros((len(retention_rates), len(offer_costs)))
for i, rr in enumerate(retention_rates):
    for j, oc in enumerate(offer_costs):
        heatmap_data[i, j] = (gross_per_customer * rr - oc).sum()

hm_df = pd.DataFrame(
    heatmap_data / 1000,
    index=[f'{r:.0%}' for r in retention_rates],
    columns=[f'${c}' for c in offer_costs]
)

fig, ax = plt.subplots(figsize=(11, 6))
sns.heatmap(
    hm_df, annot=True, fmt='.0f', cmap='RdYlGn', center=0,
    linewidths=0.5, ax=ax, annot_kws={'size': 9}
)
ax.set_title('Net ROI Sensitivity ($k) — rows: retention rate, cols: offer cost', fontsize=12)
ax.set_xlabel('Offer Cost (USD equiv per customer)')
ax.set_ylabel('Retention Rate')

# Outline near-breakeven cells (|ROI| < 10k)
for i in range(len(retention_rates)):
    for j in range(len(offer_costs)):
        if abs(heatmap_data[i, j]) < 10_000:
            ax.add_patch(plt.Rectangle((j, i), 1, 1, fill=False, edgecolor='black', lw=2.5))

plt.tight_layout()
plt.savefig(OUTPUTS / '13_roi_sensitivity.png', dpi=150, bbox_inches='tight')
plt.close()
print('Saved 13_roi_sensitivity.png')
print('Cells outlined in black = near breakeven (|ROI| < $10k)')

Saved 13_roi_sensitivity.png
Cells outlined in black = near breakeven (|ROI| < $10k)


## Section 4 — Segment Prioritisation

In [7]:
# Load segment labels — use is_string_dtype to handle ArrowStringArray
features = pd.read_parquet(OUTPUTS / 'kkbox_features.parquet')
features.columns = [c.replace(' ', '_').replace('-', '_') for c in features.columns]

seg_col = None
for col in features.columns:
    if 'segment' in col.lower() and pd.api.types.is_string_dtype(features[col]):
        seg_col = col
        break

if seg_col is None:
    raise ValueError('Cannot find segment label column in kkbox_features.parquet')

print(f'Segment column: {seg_col}')
print(features[seg_col].value_counts())

seg_lookup = features[['msno', seg_col]].rename(columns={seg_col: 'segment'})
df_seg = df.merge(seg_lookup, on='msno', how='left')
df_seg['segment'] = df_seg['segment'].fillna('Unknown').astype(str)
print(f'\nMerge coverage: {df_seg["segment"].ne("Unknown").mean()*100:.1f}%')
print('Segments in test set:', df_seg['segment'].value_counts().to_dict())

Segment column: segment_label
segment_label
Casual Listeners    421858
Power Users         274255
At-Risk             157106
Dormant             117741
Name: count, dtype: int64



Merge coverage: 100.0%
Segments in test set: {'Casual Listeners': 82893, 'At-Risk': 36757, 'Power Users': 4738, 'Dormant': 884}


In [8]:
# For each segment: expected ROI targeting all predicted churners in that segment
segment_roi = []
for seg, grp in df_seg.groupby('segment'):
    seg_churners = grp[grp['is_predicted_churn'] == 1].copy()
    n_seg = len(seg_churners)
    if n_seg == 0:
        continue
    ev_seg = (
        seg_churners['churn_prob'] * seg_churners['predicted_clv'] * RETENTION_RATE
        - OFFER_COST
    ).sum()
    segment_roi.append({
        'segment':              seg,
        'n_predicted_churners': n_seg,
        'mean_clv':             seg_churners['predicted_clv'].mean(),
        'mean_churn_prob':      seg_churners['churn_prob'].mean(),
        'net_roi':              ev_seg,
        'campaign_cost':        n_seg * OFFER_COST,
        'roi_pct':              ev_seg / (n_seg * OFFER_COST) * 100
    })

seg_roi_df = pd.DataFrame(segment_roi).sort_values('net_roi', ascending=False)
print('Segment ROI summary:')
print(seg_roi_df.to_string(index=False, float_format=lambda x: f'{x:,.1f}'))

Segment ROI summary:
         segment  n_predicted_churners  mean_clv  mean_churn_prob     net_roi  campaign_cost  roi_pct
Casual Listeners                 11095     916.2              0.8 1,560,636.1       88,760.0  1,758.3
         At-Risk                  4603     814.7              0.8   584,475.4       36,824.0  1,587.2
     Power Users                   536     708.9              0.8    58,672.9        4,288.0  1,368.3
         Dormant                    75     397.3              0.9     4,987.1          600.0    831.2


In [9]:
# Bar chart: expected ROI per segment
colours = ['#2ecc71' if v >= 0 else '#e74c3c' for v in seg_roi_df['net_roi']]

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.bar(
    seg_roi_df['segment'], seg_roi_df['net_roi'] / 1000,
    color=colours, edgecolor='white', linewidth=0.8
)
ax.axhline(0, color='black', linewidth=0.8, linestyle='--')
ax.set_title(
    'Expected Net ROI per Segment\n'
    '(all predicted churners, retention_rate=25%, offer_cost=$8)',
    fontsize=11
)
ax.set_xlabel('Segment')
ax.set_ylabel('Net ROI ($k)')

for bar, val in zip(bars, seg_roi_df['net_roi']):
    offset = 1 if val >= 0 else -3
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + offset,
        f'${val/1000:.0f}k',
        ha='center', va='bottom', fontsize=9, fontweight='bold'
    )

plt.tight_layout()
plt.savefig(OUTPUTS / '13_segment_roi.png', dpi=150, bbox_inches='tight')
plt.close()
print('Saved 13_segment_roi.png')

best = seg_roi_df.iloc[0]
print(f"\nRecommendation: Prioritise '{best['segment']}' first.")
print(f"  {best['n_predicted_churners']:,} predicted churners, "
      f"mean CLV=${best['mean_clv']:.0f}, net ROI=${best['net_roi']:,.0f}")

Saved 13_segment_roi.png

Recommendation: Prioritise 'Casual Listeners' first.
  11,095 predicted churners, mean CLV=$916, net ROI=$1,560,636


## Save ROI Summary

In [10]:
overall_row = pd.DataFrame([{
    'segment':              'ALL (high-value target)',
    'n_predicted_churners': n_target,
    'mean_clv':             target['predicted_clv'].mean(),
    'mean_churn_prob':      target['churn_prob'].mean(),
    'net_roi':              net_roi,
    'campaign_cost':        total_campaign_cost,
    'roi_pct':              roi_pct
}])

roi_summary = pd.concat([overall_row, seg_roi_df], ignore_index=True)
roi_summary.to_csv(OUTPUTS / 'roi_summary.csv', index=False)
print('Saved roi_summary.csv')
print(roi_summary.to_string(index=False, float_format=lambda x: f'{x:,.2f}'))

print('\n=== Notebook 13 complete ===')
print('Outputs: roi_summary.csv, 13_roi_sensitivity.png, 13_segment_roi.png')

Saved roi_summary.csv
                segment  n_predicted_churners  mean_clv  mean_churn_prob      net_roi  campaign_cost  roi_pct
ALL (high-value target)                  2869  2,729.07             0.65 1,257,503.41      22,952.00 5,478.84
       Casual Listeners                 11095    916.20             0.82 1,560,636.11      88,760.00 1,758.27
                At-Risk                  4603    814.68             0.84   584,475.40      36,824.00 1,587.21
            Power Users                   536    708.87             0.84    58,672.93       4,288.00 1,368.31
                Dormant                    75    397.30             0.87     4,987.13         600.00   831.19

=== Notebook 13 complete ===
Outputs: roi_summary.csv, 13_roi_sensitivity.png, 13_segment_roi.png
